In [1]:
!pip install -q dscript h5py pandas torch tqdm biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 MB 7.1 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 38.2 MB/s eta 0:00:00a 0:00:01


In [ ]:
# PREPROCESSING .H5 CON ESM-2 (SIZE 320-480)
import torch
import h5py
import numpy as np
from Bio import SeqIO
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModel

DATA_DIR = "/kaggle/input/datasets/dom3n1co/dataset1"
FASTA_FILE = f"{DATA_DIR}/cdhit_ready_yeast_800.fasta"
OUTPUT_H5 = "esm2_yeast_embeddings_480.h5"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Caricamento del modello ESM-2 (Versione leggera: 8M parametri, embedding size: 320)
#model_id = "facebook/esm2_t6_8M_UR50D"
model_id = "facebook/esm2_t12_35M_UR50D" # Per 480 dimensioni

print(f"Caricamento tokenizer e modello {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(device)
model.eval()

# Leggi il FASTA
sequences = {rec.id: str(rec.seq) for rec in SeqIO.parse(FASTA_FILE, "fasta")}

with h5py.File(OUTPUT_H5, "w") as h5out:
    for prot_id, seq in tqdm(sequences.items(), desc="Generando embeddings ESM-2"):
        
        # ESM-2 ha un limite massimo di token (solitamente 1024 o 1022 amminoacidi).
        # Se hai proteine più lunghe nel dataset, le tronchiamo per evitare crash.
        seq = seq[:1022] 
        
        with torch.no_grad():
            # Tokenizzazione
            inputs = tokenizer(seq, return_tensors="pt", add_special_tokens=True).to(device)
            
            # Forward pass per ottenere gli hidden states
            outputs = model(**inputs)
            
            # last_hidden_state ha shape: (1, L+2, 320) per via dei token [CLS] e [EOS]
            embeddings = outputs.last_hidden_state
            
            # Rimuoviamo la batch dimension e i token speciali inizio/fine
            # Nuova shape: (L, 320)
            prot_emb = embeddings[0, 1:-1, :]
            
        # Salva come numpy array su CPU
        h5out.create_dataset(prot_id, data=prot_emb.cpu().numpy())

print(f"Embedding ESM-2 salvati con successo in {OUTPUT_H5}!")

In [3]:
import torch
import torch.nn as nn
import h5py
import numpy as np
import pandas as pd
import dscript
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    precision_recall_curve,
    roc_auc_score,
    auc,
    precision_score,
    recall_score
)
import warnings
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIGURAZIONE ESPERIMENTI
# ─────────────────────────────────────────────
#DATA_DIR      = "/kaggle/input/datasets/dom3n1co/dataset1"
#EMBEDDINGS_H5 = "/kaggle/working/esm2_yeast_embeddings_320.h5"
#TRAIN_FILE    = f"{DATA_DIR}/dscript_train.csv"
#TEST_FILE     = f"{DATA_DIR}/dscript_test.csv"
#LOC_FILE = f"{DATA_DIR}/unified_protein_filter_ds.csv"

CONFIGS = [
    {"W_BCE": 0.60, "W_MAG": 0.20, "name": "config_1"},
    {"W_BCE": 0.50, "W_MAG": 0.30, "name": "config_2"},
    {"W_BCE": 0.40, "W_MAG": 0.25, "name": "config_3"}
]

DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR      = "/kaggle/input/datasets/dom3n1co/dataset1"
EMBEDDINGS_H5 = "/kaggle/working/esm2_yeast_embeddings_480.h5"
TRAIN_FILE    = f"{DATA_DIR}/dscript_train.csv"
TEST_FILE     = f"{DATA_DIR}/dscript_test.csv"
LOC_FILE = f"{DATA_DIR}/unified_protein_filter_ds.csv"
EPOCHS        = 15
BATCH_SIZE    = 16
LEARNING_RATE = 1e-4
THRESHOLD     = 0.5

print(f"Device impostato su: {DEVICE}")

Device impostato su: cuda


In [4]:
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import torch
import h5py
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

class YeastPPIDataset(Dataset):
    def __init__(self, csv_file, h5_file, loc_csv_file=None, fraction=1.0):
        # 1. Caricamento Coppie
        sep = '\t' if str(csv_file).endswith('.tsv') else ','
        self.pairs = pd.read_csv(csv_file, sep=sep)
        if len(self.pairs.columns) >= 3 and 'protein1' not in self.pairs.columns:
            self.pairs.columns = ['protein1', 'protein2', 'label'] + list(self.pairs.columns[3:])

        # 2. Caricamento Zone (Opzionale per il Test Set)
        self.locations = {}
        if loc_csv_file is not None:
            print("Caricamento zone subcellulari in RAM...")
            loc_df = pd.read_csv(loc_csv_file)
            zone_cols = ['nucleus', 'cytoplasm', 'mitochondrion', 'plasma_membrane', 'endoplasmic_reticulum', 'golgi_apparatus', 'vacuole']
            for _, row in loc_df.iterrows():
                prot_id = row['Entry']
                loc_tensor = torch.tensor(row[zone_cols].values.astype(float), dtype=torch.float32)
                self.locations[prot_id] = loc_tensor
        else:
            print("Nessun file delle zone fornito: si procederà in modalità Test/Inference (location a zero).")

        # 3. Caricamento Embeddings
        self.embeddings = {}
        print("Caricamento massivo degli embedding in RAM...")
        with h5py.File(h5_file, 'r') as f:
            available = set(f.keys())
            for prot in tqdm(available, desc="Loading H5 to RAM", leave=False):
                raw_data = f[prot][()]
                if len(raw_data.shape) == 3 and raw_data.shape[0] == 1:
                    data = raw_data[0]
                else:
                    data = raw_data
                self.embeddings[prot] = torch.tensor(data, dtype=torch.float32).to(DEVICE)

        # 4. Filtraggio
        mask = (self.pairs['protein1'].isin(available) & self.pairs['protein2'].isin(available))
        self.pairs = self.pairs[mask].reset_index(drop=True)

        if fraction < 1.0:
            self.pairs = (
                self.pairs.groupby('label', group_keys=False)
                .apply(lambda g: g.sample(frac=fraction, random_state=42))
                .reset_index(drop=True)
            )

        print(f"  Coppie disponibili: {len(self.pairs)}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        row = self.pairs.iloc[idx]
        prot1, prot2 = row['protein1'], row['protein2']
        
        # Se una proteina non ha dati di zona nel CSV, restituiamo un vettore di zeri
        loc1 = self.locations.get(prot1, torch.zeros(7, dtype=torch.float32))
        loc2 = self.locations.get(prot2, torch.zeros(7, dtype=torch.float32))
        
        return (
            self.embeddings[prot1], 
            self.embeddings[prot2], 
            loc1, 
            loc2, 
            torch.tensor(float(row['label']), dtype=torch.float32)
        )

def collate_fn_padded(batch):
    embs1, embs2, locs1, locs2, labels = zip(*batch)
    
    embs1_padded = pad_sequence(embs1, batch_first=True, padding_value=0.0)
    embs2_padded = pad_sequence(embs2, batch_first=True, padding_value=0.0)
    
    lengths1 = torch.tensor([len(x) for x in embs1])
    lengths2 = torch.tensor([len(x) for x in embs2])
    
    mask1 = torch.arange(embs1_padded.shape[1]).expand(len(embs1), -1) < lengths1.unsqueeze(1)
    mask2 = torch.arange(embs2_padded.shape[1]).expand(len(embs2), -1) < lengths2.unsqueeze(1)
    
    return embs1_padded, embs2_padded, mask1, mask2, torch.stack(locs1), torch.stack(locs2), torch.stack(labels)

In [5]:
import torch
import torch.nn as nn

class CustomAttentionPPI_Auto(nn.Module):
    def __init__(self, embed_dim=100, hidden_dim=50, num_heads=5, gamma=0.0):
        super().__init__()
        self.gamma = gamma
        
        # --- MODIFICA 1: Matrice di Affinità Dinamica Apprendibile ---
        # Partiamo con una diagonale forte (eye) + rumore per rompere la simmetria iniziale
        self.affinity_matrix = nn.Parameter(torch.eye(7) + 0.1 * torch.randn(7, 7))
        
        # --- UPGRADE 4: Impostazioni Multi-Head ---
        self.num_heads = num_heads
        assert hidden_dim % num_heads == 0, "hidden_dim deve essere divisibile per num_heads"
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** 0.5
        
        # --- UPGRADE 2: Convoluzioni 1D Dilatate in Parallelo ---
        self.conv_d1 = nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=4, dilation=4)
        self.relu_conv = nn.ReLU()
        
        self.layer_norm = nn.LayerNorm(embed_dim)
        
        # Proiezioni Non-Lineari (MLP)
        self.q_net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.k_net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, z1, z2, mask1, mask2):
        # z1_local / z2_local calcolo (invariato)
        z1_t = z1.transpose(1, 2)
        z2_t = z2.transpose(1, 2)
        
        z1_local = self.relu_conv(self.conv_d1(z1_t) + self.conv_d2(z1_t) + self.conv_d4(z1_t)).transpose(1, 2)
        z2_local = self.relu_conv(self.conv_d1(z2_t) + self.conv_d2(z2_t) + self.conv_d4(z2_t)).transpose(1, 2)
        
        z1 = z1 + self.layer_norm(z1_local)
        z2 = z2 + self.layer_norm(z2_local)
        
        # --- MODIFICA 2: Cross-Attention Bidirezionale ---
        # Calcoliamo Query e Key per ENTRAMBE le proteine
        Q1 = self.q_net(z1) # z1 come "domanda"
        K1 = self.k_net(z1) # z1 come "risposta"
        Q2 = self.q_net(z2) # z2 come "domanda"
        K2 = self.k_net(z2) # z2 come "risposta"
        
        B, N, _ = Q1.shape
        _, M, _ = Q2.shape
        
        # Split per la Multi-Head Attention
        Q1 = Q1.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K1 = K1.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        Q2 = Q2.view(B, M, self.num_heads, self.head_dim).transpose(1, 2)
        K2 = K2.view(B, M, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 1. z1 interroga z2 (Shape: Batch, Heads, N, M)
        scores_1 = torch.matmul(Q1, K2.transpose(-1, -2)) / self.scale 
        
        # 2. z2 interroga z1 (Shape: Batch, Heads, M, N)
        scores_2 = torch.matmul(Q2, K1.transpose(-1, -2)) / self.scale 
        
        # Fondiamo le due mappe per simmetria: trasponiamo scores_2 per allinearlo a (N, M) e facciamo la media
        interaction_scores_heads = (scores_1 + scores_2.transpose(-1, -2)) / 2.0
        
        # Fondiamo le teste calcolando la media matematica
        interaction_scores = interaction_scores_heads.mean(dim=1) # (Batch, N, M)
        
        # Applichiamo la maschera per ignorare il padding
        mask_2d = mask1.unsqueeze(2) & mask2.unsqueeze(1)
        interaction_scores = interaction_scores.masked_fill(~mask_2d, float('-inf'))
        
        # Mappa di Contatto finale
        C = torch.sigmoid(interaction_scores)
        
        # Pooling Vettorializzato
        C_masked = C * mask_2d.float()
        valid_counts = mask_2d.sum(dim=(1, 2)).float()
        valid_counts = torch.clamp(valid_counts, min=1e-6)
        
        mu = C_masked.sum(dim=(1, 2)) / valid_counts
        
        diff_sq = ((C_masked - mu.view(-1, 1, 1)) ** 2) * mask_2d.float()
        sigma = torch.sqrt(torch.clamp(diff_sq.sum(dim=(1, 2)) / valid_counts, min=1e-9))
        
        threshold = mu + (self.gamma * sigma)
        
        Q_raw = C_masked - threshold.view(-1, 1, 1)
        Q_raw = Q_raw.masked_fill(~mask_2d, 0.0) 
        Q_filtered = torch.relu(Q_raw)
        
        p_raw = Q_filtered.sum(dim=(1, 2)) / (torch.sign(Q_filtered).sum(dim=(1, 2)) + 1)
        
        # Logistic Activation
        eta = 20.0
        x0 = 0.5
        phat_batch = torch.sigmoid(eta * (p_raw - x0))
        phat_batch = torch.clamp(phat_batch, 1e-7, 1.0 - 1e-7) 
        
        # Generiamo la versione normalizzata e simmetrica della matrice di affinità
        A_sym = torch.sigmoid(self.affinity_matrix + self.affinity_matrix.T)
            
        return phat_batch, C, mask_2d, A_sym

def custom_loss_auto(phat, label, C, mask_2d, loc1, loc2, A_sym, w_bce=0.50, w_mag=0.35):
    assert (w_bce + w_mag) <= 1.0
    w_spatial = 1.0 - (w_bce + w_mag)
    bce = nn.BCELoss()(phat, label)
    valid_contacts = C[mask_2d]
    mag = torch.mean(valid_contacts) if len(valid_contacts) > 0 else torch.tensor(0.0, device=phat.device)
    
    S = torch.sum(torch.mm(loc1, A_sym) * loc2, dim=1)
    has_location = (loc1.sum(dim=1) > 0) & (loc2.sum(dim=1) > 0)
    penalty_mask = (S < 0.1) & has_location & (label == 0)
    spatial_penalty = torch.mean(phat * penalty_mask.float())
    return (w_bce * bce) + (w_mag * mag) + (w_spatial * spatial_penalty)

In [6]:
import torch
import torch.nn as nn

# Matrice di Affinità Biologica (7x7)
# 1 = comunicano biologicamente, 0 = isolate
AFFINITY_MATRIX = torch.tensor([
    [1., 1., 0., 0., 0., 0., 0.], # Nucleo
    [1., 1., 1., 1., 1., 1., 1.], # Citoplasma
    [0., 1., 1., 0., 0., 0., 0.], # Mitocondrio
    [0., 1., 0., 1., 0., 1., 0.], # Membrana
    [0., 1., 0., 0., 1., 1., 0.], # ER
    [0., 1., 0., 1., 1., 1., 1.], # Golgi
    [0., 1., 0., 0., 0., 1., 1.]  # Vacuolo
])

class CustomAttentionPPI(nn.Module):
    def __init__(self, embed_dim=100, hidden_dim=50, num_heads=5, gamma=0.0):
        super().__init__()
        self.gamma = gamma
        
        # --- UPGRADE 4: Impostazioni Multi-Head ---
        self.num_heads = num_heads
        assert hidden_dim % num_heads == 0, "hidden_dim deve essere divisibile per num_heads"
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** 0.5
        
        # --- UPGRADE 2: Convoluzioni 1D Dilatate in Parallelo ---
        # Invece di una sola finestra, guardiamo a distanze diverse (vicino, medio, lontano)
        self.conv_d1 = nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=4, dilation=4)
        self.relu_conv = nn.ReLU()
        
        # Manteniamo la LayerNorm stabilizzante
        self.layer_norm = nn.LayerNorm(embed_dim)
        
        # Proiezioni Non-Lineari (MLP)
        self.q_net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.k_net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, z1, z2, mask1, mask2):
        # z1/z2 original shape: (Batch, SeqLen, EmbedDim)
        
        # --- UPGRADE 2: Esecuzione delle Convoluzioni Dilatate ---
        z1_t = z1.transpose(1, 2)
        z2_t = z2.transpose(1, 2)
        
        # Sommiamo i 3 "campi visivi" e applichiamo ReLU
        z1_local = self.relu_conv(self.conv_d1(z1_t) + self.conv_d2(z1_t) + self.conv_d4(z1_t)).transpose(1, 2)
        z2_local = self.relu_conv(self.conv_d1(z2_t) + self.conv_d2(z2_t) + self.conv_d4(z2_t)).transpose(1, 2)
        
        # Normalizzazione e Residual Connection
        z1_local = self.layer_norm(z1_local)
        z2_local = self.layer_norm(z2_local)
        z1 = z1 + z1_local
        z2 = z2 + z2_local
        
        # Proiezione in Q e K
        Q = self.q_net(z1) # (Batch, N, hidden_dim)
        K = self.k_net(z2) # (Batch, M, hidden_dim)
        
        B, N, _ = Q.shape
        _, M, _ = K.shape
        
        # --- UPGRADE 4: Split per la Multi-Head Attention ---
        # Dividiamo l'hidden_dim in più "teste" indipendenti
        Q = Q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2) # (Batch, Heads, N, HeadDim)
        K = K.view(B, M, self.num_heads, self.head_dim).transpose(1, 2) # (Batch, Heads, M, HeadDim)
        
        # Calcoliamo le mappe di attenzione per ogni singola testa
        # torch.matmul gestisce automaticamente le 4 dimensioni
        interaction_scores_heads = torch.matmul(Q, K.transpose(-1, -2)) / self.scale # (Batch, Heads, N, M)
        
        # Fondiamo le teste calcolando la media matematica per ottenere la mappa finale
        interaction_scores = interaction_scores_heads.mean(dim=1) # (Batch, N, M)
        
        # Applichiamo la maschera per ignorare il padding
        mask_2d = mask1.unsqueeze(2) & mask2.unsqueeze(1)
        interaction_scores = interaction_scores.masked_fill(~mask_2d, float('-inf'))
        
        # Mappa di Contatto finale
        C = torch.sigmoid(interaction_scores)
        
        # Pooling Vettorializzato
        C_masked = C * mask_2d.float()
        valid_counts = mask_2d.sum(dim=(1, 2)).float()
        valid_counts = torch.clamp(valid_counts, min=1e-6)
        
        mu = C_masked.sum(dim=(1, 2)) / valid_counts
        
        diff_sq = ((C_masked - mu.view(-1, 1, 1)) ** 2) * mask_2d.float()
        sigma = torch.sqrt(torch.clamp(diff_sq.sum(dim=(1, 2)) / valid_counts, min=1e-9))
        
        threshold = mu + (self.gamma * sigma)
        
        # Isoliamo solo i contatti forti
        Q_raw = C_masked - threshold.view(-1, 1, 1)
        Q_raw = Q_raw.masked_fill(~mask_2d, 0.0) 
        Q_filtered = torch.relu(Q_raw)
        
        # Calcolo raw probability
        p_raw = Q_filtered.sum(dim=(1, 2)) / (torch.sign(Q_filtered).sum(dim=(1, 2)) + 1)
        
        # Logistic Activation
        eta = 20.0
        x0 = 0.5
        phat_batch = torch.sigmoid(eta * (p_raw - x0))
        phat_batch = torch.clamp(phat_batch, 1e-7, 1.0 - 1e-7) 
            
        return phat_batch, C, mask_2d

# Loss per il modello Statico (quello con la matrice hardcodata)
def custom_loss_spatial(phat, label, C, mask_2d, loc1, loc2, w_bce=0.50, w_mag=0.35):
    assert (w_bce + w_mag) <= 1.0
    w_spatial = 1.0 - (w_bce + w_mag)
    bce = nn.BCELoss()(phat, label)
    valid_contacts = C[mask_2d]
    mag = torch.mean(valid_contacts) if len(valid_contacts) > 0 else torch.tensor(0.0, device=phat.device)
    
    affinity = AFFINITY_MATRIX.to(loc1.device)
    S = torch.sum(torch.mm(loc1, affinity) * loc2, dim=1)
    has_location = (loc1.sum(dim=1) > 0) & (loc2.sum(dim=1) > 0)
    penalty_mask = (S == 0) & has_location & (label == 0)
    spatial_penalty = torch.mean(phat * penalty_mask.float())
    return (w_bce * bce) + (w_mag * mag) + (w_spatial * spatial_penalty)

In [ ]:
import random
import os
import numpy as np
import torch

# --- IMPOSTAZIONE SEED BASE ---
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
# ------------------------------

DATA_DIR      = "/kaggle/input/datasets/dom3n1co/dataset1"
EMBEDDINGS_H5 = "/kaggle/working/esm2_yeast_embeddings_480.h5"
TRAIN_FILE    = f"{DATA_DIR}/dscript_train.csv"
TEST_FILE     = f"{DATA_DIR}/dscript_test.csv"
LOC_FILE = f"{DATA_DIR}/unified_protein_filter_ds.csv"
FRACTION = 1   # 1.0 = tutto il dataset

print("--- Caricamento dataset ---")
train_dataset = YeastPPIDataset(TRAIN_FILE, EMBEDDINGS_H5, LOC_FILE, fraction=FRACTION)
test_dataset  = YeastPPIDataset(TEST_FILE,  EMBEDDINGS_H5, LOC_FILE, fraction=FRACTION)

print("\n--- Creazione DataLoader Vettorializzati ---")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn_padded)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_padded)

print("\n--- Inizializzazione Modello Custom ---")
model = CustomAttentionPPI(embed_dim=480, hidden_dim=100, num_heads=5, gamma=0.0).to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametri addestrabili: {trainable_params:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [ ]:
import numpy as np
import torch
from tqdm.notebook import tqdm
from sklearn.metrics import precision_recall_curve, roc_auc_score, auc, precision_score, recall_score

import numpy as np
import torch
from tqdm.notebook import tqdm
from sklearn.metrics import precision_recall_curve, roc_auc_score, auc, precision_score, recall_score

# 1. Aggiungi **model_kwargs qui
def run_model_training(model_class, loss_fn, is_auto, configs, train_loader, device, epochs, lr, threshold=0.5, **model_kwargs):
    """
    Funzione universale per addestrare varianti del modello CustomAttentionPPI.
    """
    suffix = "_auto" if is_auto else ""
    print(f"\n=== Inizio Training Multiplo per {model_class.__name__} ({len(configs)} configurazioni) ===")

    for conf in configs:
        current_w_bce = conf["W_BCE"]
        current_w_mag = conf["W_MAG"]
        model_name = conf["name"]
        
        print(f"\n" + "="*50)
        print(f"=== AVVIO TRAINING: {model_name}{suffix} (W_BCE={current_w_bce}, W_MAG={current_w_mag}) ===")
        print("="*50)
        
        # 2. Inizializzazione Modello dinamica pulita
        model = model_class(**model_kwargs).to(device)
        
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

        best_aupr = 0.0

        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            all_preds, all_labels = [], []
            
            progress_bar = tqdm(train_loader, desc=f"[{model_name}{suffix}] Epoch {epoch+1}/{epochs}")
            
            for embs1_pad, embs2_pad, mask1, mask2, locs1, locs2, labels in progress_bar:
                embs1_pad, embs2_pad = embs1_pad.to(device), embs2_pad.to(device)
                mask1, mask2 = mask1.to(device), mask2.to(device)
                locs1, locs2 = locs1.to(device), locs2.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                # --- FORWARD PASS DINAMICO ---
                if is_auto:
                    phat, C, mask_2d, A_sym = model(embs1_pad, embs2_pad, mask1, mask2)
                    loss = loss_fn(phat, labels, C, mask_2d, locs1, locs2, A_sym, w_bce=current_w_bce, w_mag=current_w_mag)
                else:
                    phat, C, mask_2d = model(embs1_pad, embs2_pad, mask1, mask2)
                    loss = loss_fn(phat, labels, C, mask_2d, locs1, locs2, w_bce=current_w_bce, w_mag=current_w_mag)
                
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
                
                all_preds.extend(phat.detach().cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
            # --- METRICHE FINE EPOCA ---
            avg_epoch_loss = epoch_loss / len(train_loader)
            
            y_true = np.array(all_labels)
            y_pred = np.array(all_preds)
            y_pred_bin = (y_pred >= threshold).astype(int)
            
            try:
                epoch_auroc = roc_auc_score(y_true, y_pred)
            except ValueError:
                epoch_auroc = 0.0
                
            precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_pred)
            epoch_aupr = auc(recall_curve, precision_curve)
            epoch_precision = precision_score(y_true, y_pred_bin, zero_division=0)
            epoch_recall = recall_score(y_true, y_pred_bin, zero_division=0)
            
            scheduler.step(epoch_aupr)
            
            print(f"\n--- Risultati Epoch {epoch+1} ({model_name}{suffix}) ---")
            print(f"Loss:      {avg_epoch_loss:.4f}")
            print(f"AUPR:      {epoch_aupr:.4f}")
            print(f"Precision: {epoch_precision:.4f}")
            print(f"Recall:    {epoch_recall:.4f}")
            print(f"AUROC:     {epoch_auroc:.4f}\n")

            # --- SALVATAGGIO DINAMICO ---
            best_model_filename = f"best_{model_name}{suffix}.pt"
            checkpoint_filename = f"latest_checkpoint_{model_name}{suffix}.pt"
            
            if epoch_aupr > best_aupr:
                best_aupr = epoch_aupr
                torch.save(model.state_dict(), best_model_filename)
                print(f"  [!] Nuovo Best Model salvato in '{best_model_filename}'! (AUPR record: {best_aupr:.4f})")
                
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_aupr': best_aupr
            }
            torch.save(checkpoint, checkpoint_filename)
        
    print(f"\nAddestramento Multiplo per {model_class.__name__} completato con successo!")

In [ ]:
run_model_training(
    model_class=CustomAttentionPPI, 
    loss_fn=custom_loss_spatial, 
    is_auto=False, 
    configs=CONFIGS, 
    train_loader=train_loader, 
    device=DEVICE, 
    epochs=EPOCHS, 
    lr=LEARNING_RATE,
    # --- Parametri specifici del modello passati tramite kwargs ---
    embed_dim=480,
    hidden_dim=100,  # <-- Impostato a 100 qui
    num_heads=5,
    gamma=0.0
)

In [7]:
import torch
import numpy as np
import dscript
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, precision_score, recall_score

THRESHOLD = 0.5

# ─────────────────────────────────────────────
# 1. FUNZIONI DI VALUTAZIONE
# ─────────────────────────────────────────────
def calculate_metrics(y_true, y_pred, threshold=0.5):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_pred_bin = (y_pred >= threshold).astype(int)
    
    try:
        auroc = roc_auc_score(y_true, y_pred)
    except ValueError:
        auroc = 0.0
        
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_pred)
    aupr = auc(recall_curve, precision_curve)
    precision = precision_score(y_true, y_pred_bin, zero_division=0)
    recall = recall_score(y_true, y_pred_bin, zero_division=0)
    
    return {"AUPR": aupr, "Precision": precision, "Recall": recall, "AUROC": auroc}

def evaluate_baseline(base_model, dataset, device):
    """ Valuta D-SCRIPT originale elaborando una coppia alla volta (no padding) """
    base_model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for i in tqdm(range(len(dataset)), desc="Eval Baseline (Zero-Shot)"):
            
            # --- IL FIX È QUI ---
            # Decomprimiamo 5 valori ignorando le location (i due underscore)
            emb1, emb2, _, _, label = dataset[i]
            # --------------------
            
            emb1, emb2 = emb1.unsqueeze(0).to(device), emb2.unsqueeze(0).to(device)
            
            # Bypassiamo il projection layer (gli embedding sono già a 100 dim)
            C = base_model.contact(emb1, emb2)
            yhat = base_model.maxPool(C) if base_model.do_pool else C
            
            mu = torch.mean(yhat)
            sigma = torch.var(yhat)
            Q = torch.relu(yhat - mu - (base_model.gamma * sigma))
            phat = torch.sum(Q) / (torch.sum(torch.sign(Q)) + 1)
            if base_model.do_sigmoid:
                phat = base_model.activation(phat).squeeze()
                
            all_preds.append(phat.cpu().item())
            all_labels.append(label.item())
            
    return calculate_metrics(all_labels, all_preds, THRESHOLD)

def evaluate_custom(model, dataloader, device):
    """ 
    Valuta modelli custom con output variabili. 
    Funziona sia con modelli che restituiscono 3 valori che con modelli da 4.
    """
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        # Caricamento dati (7 valori: embs x2, masks x2, locs x2, labels)
        for embs1_pad, embs2_pad, mask1, mask2, _, _, labels in tqdm(dataloader, desc="Eval Model"):
            embs1_pad, embs2_pad = embs1_pad.to(device), embs2_pad.to(device)
            mask1, mask2, labels = mask1.to(device), mask2.to(device), labels.to(device)
            
            # --- SOLUZIONE: Catturiamo phat e ignoriamo tutto il resto con *_ ---
            # Questo prende il primo valore e mette tutti gli altri in una lista chiamata '_'
            outputs = model(embs1_pad, embs2_pad, mask1, mask2)
            phat = outputs[0] 
            
            all_preds.extend(phat.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return calculate_metrics(all_labels, all_preds, THRESHOLD)

In [8]:
def prepare_test_data(test_file, esm2_h5, dscript_h5, loc_file, batch_size, fraction=1.0):
    """
    Carica i dataset e crea il DataLoader per il modello custom.
    """
    print(f"--- Caricamento dataset: {test_file.split('/')[-1]} ---")
    
    # Dataset per ESM2 (Modello Custom)
    test_dataset_esm2 = YeastPPIDataset(test_file, esm2_h5, loc_file, fraction=fraction)
    test_loader_esm2 = DataLoader(
        test_dataset_esm2, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_padded
    )
    
    # Dataset per D-SCRIPT (Baseline)
    test_dataset_dscript = YeastPPIDataset(test_file, dscript_h5, loc_file, fraction=fraction)
    
    print(f"Done. Dataset size: {len(test_dataset_esm2)}")
    return test_dataset_esm2, test_loader_esm2, test_dataset_dscript

def run_full_evaluation(organism_name, test_dataset_dscript, test_loader_esm2, configs, device):
    """
    Esegue la baseline, valuta le configurazioni Auto e Spatial e stampa il confronto.
    """
    # 1. VALUTAZIONE BASELINE
    print(f"\n--- Caricamento Baseline (D-SCRIPT human_v1) per {organism_name} ---")
    base_model = dscript.pretrained.get_pretrained("human_v1").to(device)
    base_model.eval()
    baseline_metrics = evaluate_baseline(base_model, test_dataset_dscript, device)
    del base_model
    torch.cuda.empty_cache()

    # 2. VALUTAZIONE CONFIGURAZIONI CUSTOM
    results_auto = {}
    results_spatial = {}

    for conf in configs:
        model_name = conf["name"]
        
        # Eval Modello AUTO
        weights_path_auto = f"/kaggle/working/best_{model_name}_auto.pt"
        print(f"Valutando AUTO: {model_name}...")
        model_auto = CustomAttentionPPI_Auto(embed_dim=480, hidden_dim=50, gamma=0.0).to(device)
        model_auto.load_state_dict(torch.load(weights_path_auto, map_location=device))
        model_auto.eval()
        results_auto[model_name] = evaluate_custom(model_auto, test_loader_esm2, device)
        
        # Eval Modello SPATIAL
        weights_path_spatial = f"/kaggle/working/custom_attention_ppi_spatial_{model_name}.pt"
        print(f"Valutando SPATIAL: {model_name}...")
        model_spatial = CustomAttentionPPI(embed_dim=480, hidden_dim=50, gamma=0.0).to(device)
        model_spatial.load_state_dict(torch.load(weights_path_spatial, map_location=device))
        model_spatial.eval()
        results_spatial[model_name] = evaluate_custom(model_spatial, test_loader_esm2, device)
        
        torch.cuda.empty_cache()

    # 3. STAMPA TABELLA FINALE
    print("\n" + "="*110)
    print(f"{'Configurazione':<25} | {'AUPR':<7} | {'AUROC':<7} | {'Prec.':<7} | {'Recall':<7} | {'Δ AUPR':<8} | {'Δ AUROC':<8}")
    print("-" * 110)

    b = baseline_metrics
    print(f"{'BASELINE (D-SCRIPT)':<25} | {b['AUPR']:.4f} | {b['AUROC']:.4f} | {b['Precision']:.4f} | {b['Recall']:.4f} | {'-':<8} | {'-':<8}")

    best_name = None
    max_aupr = -1

    def print_res_row(name, m, base):
        nonlocal best_name, max_aupr
        d_aupr = m['AUPR'] - base['AUPR']
        d_auroc = m['AUROC'] - base['AUROC']
        if m['AUPR'] > max_aupr:
            max_aupr = m['AUPR']; best_name = name
        print(f"{name:<25} | {m['AUPR']:.4f} | {m['AUROC']:.4f} | {m['Precision']:.4f} | {m['Recall']:.4f} | {d_aupr:+.4f} | {d_auroc:+.4f}")

    for name, m in results_auto.items(): print_res_row(f"{name} (Auto)", m, b)
    for name, m in results_spatial.items(): print_res_row(f"{name} (Spatial)", m, b)

    print("-" * 110)
    print(f"🏆 MIGLIORE CONFIGURAZIONE TOTALE SU {organism_name.upper()}: {best_name}")
    print(f"📈 VALORE AUPR: {max_aupr:.4f}")
    print("="*110)


def run_CustomAttentionPPI_evaluation(organism_name, test_dataset_dscript, test_loader_esm2, configs, device):
    """
    Esegue la baseline, valuta ESCLUSIVAMENTE il modello CustomAttentionPPI 
    e stampa la tabella di confronto finale.
    """
    # 1. VALUTAZIONE BASELINE
    print(f"\n--- Caricamento Baseline (D-SCRIPT human_v1) per {organism_name} ---")
    base_model = dscript.pretrained.get_pretrained("human_v1").to(device)
    base_model.eval()
    baseline_metrics = evaluate_baseline(base_model, test_dataset_dscript, device)
    del base_model
    torch.cuda.empty_cache()

    # 2. VALUTAZIONE CONFIGURAZIONI CUSTOM
    results_spatial = {}

    for conf in configs:
        model_name = conf["name"]
        
        # Eval Modello CustomAttentionPPI (Spatial)
        # Assicurati che il path corrisponda a come hai salvato i pesi nel training loop
        weights_path = f"/kaggle/working/best_{model_name}.pt" 
        print(f"Valutando Modello Custom: {model_name}...")
        
        # Inizializzazione aggiornata con num_heads=5
        model_spatial = CustomAttentionPPI(embed_dim=480, hidden_dim=100, num_heads=5, gamma=0.0).to(device)
        model_spatial.load_state_dict(torch.load(weights_path, map_location=device))
        model_spatial.eval()
        
        results_spatial[model_name] = evaluate_custom(model_spatial, test_loader_esm2, device)
        
        torch.cuda.empty_cache()

    # 3. STAMPA TABELLA FINALE
    print("\n" + "="*110)
    print(f"{'Configurazione':<25} | {'AUPR':<7} | {'AUROC':<7} | {'Prec.':<7} | {'Recall':<7} | {'Δ AUPR':<8} | {'Δ AUROC':<8}")
    print("-" * 110)

    b = baseline_metrics
    print(f"{'BASELINE (D-SCRIPT)':<25} | {b['AUPR']:.4f} | {b['AUROC']:.4f} | {b['Precision']:.4f} | {b['Recall']:.4f} | {'-':<8} | {'-':<8}")

    best_name = None
    max_aupr = -1

    def print_res_row(name, m, base):
        nonlocal best_name, max_aupr
        d_aupr = m['AUPR'] - base['AUPR']
        d_auroc = m['AUROC'] - base['AUROC']
        if m['AUPR'] > max_aupr:
            max_aupr = m['AUPR']
            best_name = name
        print(f"{name:<25} | {m['AUPR']:.4f} | {m['AUROC']:.4f} | {m['Precision']:.4f} | {m['Recall']:.4f} | {d_aupr:+.4f} | {d_auroc:+.4f}")

    for name, m in results_spatial.items(): 
        print_res_row(f"{name} (Spatial)", m, b)

    print("-" * 110)
    print(f"🏆 MIGLIORE CONFIGURAZIONE TOTALE SU {organism_name.upper()}: {best_name}")
    print(f"📈 VALORE AUPR: {max_aupr:.4f}")
    print("="*110)

In [9]:
# ─────────────────────────────────────────────
# CONFRONTO TEST-SET cerevisiae 
# ─────────────────────────────────────────────

# Setup per Drosophila
TEST_FILE_CEREVISIAE = "/kaggle/input/datasets/dom3n1co/dataset1/dscript_test.csv"
# embedding
ESM2_H5_CEREVISIAE = "/kaggle/working/esm2_yeast_embeddings_480.h5"
DSCRIPT_H5_DROSO = "/kaggle/input/datasets/dom3n1co/dataset1/yeast_embeddings_projected.h5"

# Chiamata alla funzione 1
ds_esm2, loader_esm2, ds_dscript = prepare_test_data(
    TEST_FILE_CEREVISIAE, ESM2_H5_CEREVISIAE, DSCRIPT_H5_DROSO, LOC_FILE, BATCH_SIZE
)

run_full_evaluation("Cerevisiae", ds_dscript, loader_esm2, CONFIGS, DEVICE)

--- Caricamento dataset: dscript_test.csv ---
Caricamento zone subcellulari in RAM...
Caricamento massivo degli embedding in RAM...


Loading H5 to RAM:   0%|          | 0/5084 [00:00<?, ?it/s]

  Coppie disponibili: 43473
Caricamento zone subcellulari in RAM...
Caricamento massivo degli embedding in RAM...


Loading H5 to RAM:   0%|          | 0/5084 [00:00<?, ?it/s]

  Coppie disponibili: 43473
Done. Dataset size: 43473

--- Caricamento Baseline (D-SCRIPT human_v1) per Cerevisiae ---
2026-04-23 08:44:16.965 | INFO     | dscript.utils:log:49 - Downloading model human_v1 from http://cb.csail.mit.edu/cb/dscript/data/models/dscript_human_v1.pt...


...

Valutando AUTO: config_3...


Eval Model:   0%|          | 0/2718 [00:00<?, ?it/s]

Valutando SPATIAL: config_3...


Eval Model:   0%|          | 0/2718 [00:00<?, ?it/s]


Configurazione            | AUPR    | AUROC   | Prec.   | Recall  | Δ AUPR   | Δ AUROC 
--------------------------------------------------------------------------------------------------------------
BASELINE (D-SCRIPT)       | 0.4051 | 0.7812 | 0.6925 | 0.2665 | -        | -       
config_1 (Auto)           | 0.9281 | 0.9876 | 0.8545 | 0.9105 | +0.5230 | +0.2064
config_2 (Auto)           | 0.9401 | 0.9900 | 0.9491 | 0.8449 | +0.5350 | +0.2088
config_3 (Auto)           | 0.9182 | 0.9875 | 0.8719 | 0.8527 | +0.5131 | +0.2063
config_1 (Spatial)        | 0.9036 | 0.9850 | 0.8566 | 0.8369 | +0.4985 | +0.2037
config_2 (Spatial)        | 0.9227 | 0.9884 | 0.8976 | 0.8311 | +0.5176 | +0.2072
config_3 (Spatial)        | 0.8951 | 0.9829 | 0.8769 | 0.7994 | +0.4900 | +0.2016
--------------------------------------------------------------------------------------------------------------
🏆 MIGLIORE CONFIGURAZIONE TOTALE SU CEREVISIAE: config_2 (Auto)
📈 VALORE AUPR: 0.9401


In [ ]:
# DA QUI RIFACCIO IL CODICE PER TESTARLO SUL MOSCERINO

In [ ]:
# PREPROCESSING .H5 CON ESM-2 (SIZE 320)
import torch
import h5py
import numpy as np
from Bio import SeqIO
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModel

DATA_DIR = "/kaggle/input/datasets/dom3n1co/dataset1"
#FASTA_FILE = "/kaggle/input/datasets/dom3n1co/dataset1/drosophila_filter.fasta"
FASTA_FILE = "/kaggle/input/datasets/dom3n1co/dataset1/pombe_filtered.fasta"
OUTPUT_H5 = "esm2_pombe_embeddings_480.h5"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Caricamento del modello ESM-2 (Versione leggera: 8M parametri, embedding size: 320)
#model_id = "facebook/esm2_t6_8M_UR50D"
model_id = "facebook/esm2_t12_35M_UR50D" # Per 480 dimensioni

print(f"Caricamento tokenizer e modello {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(device)
model.eval()

# Leggi il FASTA
sequences = {rec.id: str(rec.seq) for rec in SeqIO.parse(FASTA_FILE, "fasta")}

with h5py.File(OUTPUT_H5, "w") as h5out:
    for prot_id, seq in tqdm(sequences.items(), desc="Generando embeddings ESM-2"):
        
        # ESM-2 ha un limite massimo di token (solitamente 1024 o 1022 amminoacidi).
        # Se hai proteine più lunghe nel dataset, le tronchiamo per evitare crash.
        seq = seq[:1022] 
        
        with torch.no_grad():
            # Tokenizzazione
            inputs = tokenizer(seq, return_tensors="pt", add_special_tokens=True).to(device)
            
            # Forward pass per ottenere gli hidden states
            outputs = model(**inputs)
            
            # last_hidden_state ha shape: (1, L+2, 320) per via dei token [CLS] e [EOS]
            embeddings = outputs.last_hidden_state
            
            # Rimuoviamo la batch dimension e i token speciali inizio/fine
            # Nuova shape: (L, 320)
            prot_emb = embeddings[0, 1:-1, :]
            
        # Salva come numpy array su CPU
        h5out.create_dataset(prot_id, data=prot_emb.cpu().numpy())

print(f"Embedding ESM-2 salvati con successo in {OUTPUT_H5}!")

In [10]:
# ─────────────────────────────────────────────
# CONFRONTO TEST-SET S. POMBE
# ─────────────────────────────────────────────

# Setup per Drosophila
TEST_FILE_POMBE = "/kaggle/input/datasets/dom3n1co/dataset1/pombe_filter_test.csv"
ESM2_H5_POMBE = "/kaggle/working/esm2_pombe_embeddings_480.h5"
DSCRIPT_H5_POMBE = "/kaggle/input/datasets/dom3n1co/dataset1/pombe_embeddings_projected.h5"

# Chiamata alla funzione 1
ds_esm2, loader_esm2, ds_dscript = prepare_test_data(
    TEST_FILE_POMBE, ESM2_H5_POMBE, DSCRIPT_H5_POMBE, LOC_FILE, BATCH_SIZE
)

--- Caricamento dataset: pombe_filter_test.csv ---
Caricamento zone subcellulari in RAM...
Caricamento massivo degli embedding in RAM...


Loading H5 to RAM:   0%|          | 0/4249 [00:00<?, ?it/s]

  Coppie disponibili: 61479
Caricamento zone subcellulari in RAM...
Caricamento massivo degli embedding in RAM...


Loading H5 to RAM:   0%|          | 0/4249 [00:00<?, ?it/s]

  Coppie disponibili: 61479
Done. Dataset size: 61479


In [11]:
# Chiamata alla funzione 2
run_full_evaluation("S. Pombe", ds_dscript, loader_esm2, CONFIGS, DEVICE)


--- Caricamento Baseline (D-SCRIPT human_v1) per S. Pombe ---


Eval Baseline (Zero-Shot):   0%|          | 0/61479 [00:00<?, ?it/s]

Valutando AUTO: config_1...


Eval Model:   0%|          | 0/3843 [00:00<?, ?it/s]

Valutando SPATIAL: config_1...


...

Valutando AUTO: config_3...


Eval Model:   0%|          | 0/3843 [00:00<?, ?it/s]

Valutando SPATIAL: config_3...


Eval Model:   0%|          | 0/3843 [00:00<?, ?it/s]


Configurazione            | AUPR    | AUROC   | Prec.   | Recall  | Δ AUPR   | Δ AUROC 
--------------------------------------------------------------------------------------------------------------
BASELINE (D-SCRIPT)       | 0.1276 | 0.6066 | 0.1380 | 0.3797 | -        | -       
config_1 (Auto)           | 0.1526 | 0.6988 | 0.1643 | 0.6114 | +0.0250 | +0.0922
config_2 (Auto)           | 0.1511 | 0.7001 | 0.1630 | 0.5380 | +0.0235 | +0.0935
config_3 (Auto)           | 0.1554 | 0.7228 | 0.1690 | 0.6735 | +0.0278 | +0.1162
config_1 (Spatial)        | 0.1576 | 0.7092 | 0.1704 | 0.6522 | +0.0300 | +0.1026
config_2 (Spatial)        | 0.1613 | 0.7223 | 0.1713 | 0.6055 | +0.0337 | +0.1157
config_3 (Spatial)        | 0.1657 | 0.7111 | 0.1753 | 0.5099 | +0.0380 | +0.1045
--------------------------------------------------------------------------------------------------------------
🏆 MIGLIORE CONFIGURAZIONE TOTALE SU S. POMBE: config_3 (Spatial)
📈 VALORE AUPR: 0.1657


In [ ]:
"""
--- Valutazione Baseline (Zero-Shot su Yeast) ---
Eval Baseline (Zero-Shot): 100%
 61479/61479 [05:15<00:00, 199.85it/s]
  AUPR: 0.1276
  Precision: 0.1380
  Recall: 0.3797
  AUROC: 0.6066

--- Caricamento Modello CustomAttention ---

--- Valutazione Modello CustomAttention (Finetuned su Yeast) ---
Eval Custom Model: 100%
 3843/3843 [00:59<00:00, 64.74it/s]
  AUPR: 0.1577
  Precision: 0.1673
  Recall: 0.6583
  AUROC: 0.7171

==================================================
=== MIGLIORAMENTO (CUSTOM ATTENTION vs BASELINE) ===
==================================================
  AUPR      : 0.1577 (Baseline: 0.1276) -> Delta: +0.0301
  Precision : 0.1673 (Baseline: 0.1380) -> Delta: +0.0293
  Recall    : 0.6583 (Baseline: 0.3797) -> Delta: +0.2786
  AUROC     : 0.7171 (Baseline: 0.6066) -> Delta: +0.1105"""

In [12]:
# ─────────────────────────────────────────────
# CONFRONTO TEST-SET drosophila
# ─────────────────────────────────────────────

# Setup per Drosophila
TEST_FILE_DROSO = "/kaggle/input/datasets/dom3n1co/dataset1/drosophila_filter_test.csv"
ESM2_H5_DROSO = "/kaggle/working/esm2_drosophila_embeddings_480.h5"
DSCRIPT_H5_DROSO = "/kaggle/input/datasets/dom3n1co/dataset1/drosophila_embeddings_projected.h5"

# Chiamata alla funzione 1
ds_esm2, loader_esm2, ds_dscript = prepare_test_data(
    TEST_FILE_DROSO, ESM2_H5_DROSO, DSCRIPT_H5_DROSO, LOC_FILE, BATCH_SIZE
)

--- Caricamento dataset: drosophila_filter_test.csv ---
Caricamento zone subcellulari in RAM...
Caricamento massivo degli embedding in RAM...


Loading H5 to RAM:   0%|          | 0/10000 [00:00<?, ?it/s]

  Coppie disponibili: 14899
Caricamento zone subcellulari in RAM...
Caricamento massivo degli embedding in RAM...


Loading H5 to RAM:   0%|          | 0/10000 [00:00<?, ?it/s]

  Coppie disponibili: 14899
Done. Dataset size: 14899


In [13]:
# Chiamata alla funzione 2
run_full_evaluation("Drosophila", ds_dscript, loader_esm2, CONFIGS, DEVICE)


--- Caricamento Baseline (D-SCRIPT human_v1) per Drosophila ---


Eval Baseline (Zero-Shot):   0%|          | 0/14899 [00:00<?, ?it/s]

Valutando AUTO: config_1...


Eval Model:   0%|          | 0/932 [00:00<?, ?it/s]

Valutando SPATIAL: config_1...


...

Valutando AUTO: config_3...


Eval Model:   0%|          | 0/932 [00:00<?, ?it/s]

Valutando SPATIAL: config_3...


Eval Model:   0%|          | 0/932 [00:00<?, ?it/s]


Configurazione            | AUPR    | AUROC   | Prec.   | Recall  | Δ AUPR   | Δ AUROC 
--------------------------------------------------------------------------------------------------------------
BASELINE (D-SCRIPT)       | 0.5631 | 0.8260 | 0.7886 | 0.3618 | -        | -       
config_1 (Auto)           | 0.5668 | 0.8392 | 0.7356 | 0.3784 | +0.0037 | +0.0132
config_2 (Auto)           | 0.6404 | 0.8871 | 0.9218 | 0.2728 | +0.0773 | +0.0611
config_3 (Auto)           | 0.6428 | 0.8882 | 0.8684 | 0.3676 | +0.0797 | +0.0622
config_1 (Spatial)        | 0.6152 | 0.8653 | 0.8043 | 0.4045 | +0.0521 | +0.0393
config_2 (Spatial)        | 0.6300 | 0.8773 | 0.8591 | 0.3133 | +0.0669 | +0.0514
config_3 (Spatial)        | 0.5746 | 0.8550 | 0.8267 | 0.2865 | +0.0115 | +0.0290
--------------------------------------------------------------------------------------------------------------
🏆 MIGLIORE CONFIGURAZIONE TOTALE SU DROSOPHILA: config_3 (Auto)
📈 VALORE AUPR: 0.6428


In [14]:
# ─────────────────────────────────────────────
# CONFRONTO TEST-SET candida_albicans
# ─────────────────────────────────────────────

# Setup per Drosophila
TEST_FILE_ALBICANS = "/kaggle/input/datasets/dom3n1co/dataset1/candida_albicans_filter_test.csv"
ESM2_H5_ALBICANS = "/kaggle/working/esm2_candida_albicans_embeddings_480.h5"
DSCRIPT_H5_ALBICANS = "/kaggle/input/datasets/dom3n1co/dataset1/candida_albicans_embeddings_projected.h5"

# Chiamata alla funzione 1
ds_esm2, loader_esm2, ds_dscript = prepare_test_data(
    TEST_FILE_ALBICANS, ESM2_H5_ALBICANS, DSCRIPT_H5_ALBICANS, LOC_FILE, BATCH_SIZE
)

--- Caricamento dataset: candida_albicans_filter_test.csv ---
Caricamento zone subcellulari in RAM...
Caricamento massivo degli embedding in RAM...


Loading H5 to RAM:   0%|          | 0/4693 [00:00<?, ?it/s]

  Coppie disponibili: 50501
Caricamento zone subcellulari in RAM...
Caricamento massivo degli embedding in RAM...


Loading H5 to RAM:   0%|          | 0/4693 [00:00<?, ?it/s]

  Coppie disponibili: 50501
Done. Dataset size: 50501


In [15]:
# Chiamata alla funzione 2
run_full_evaluation("Candida Albicans", ds_dscript, loader_esm2, CONFIGS, DEVICE)


--- Caricamento Baseline (D-SCRIPT human_v1) per Candida Albicans ---


Eval Baseline (Zero-Shot):   0%|          | 0/50501 [00:00<?, ?it/s]

Valutando AUTO: config_1...


Eval Model:   0%|          | 0/3157 [00:00<?, ?it/s]

Valutando SPATIAL: config_1...


...

Valutando AUTO: config_3...


Eval Model:   0%|          | 0/3157 [00:00<?, ?it/s]

Valutando SPATIAL: config_3...


Eval Model:   0%|          | 0/3157 [00:00<?, ?it/s]


Configurazione            | AUPR    | AUROC   | Prec.   | Recall  | Δ AUPR   | Δ AUROC 
--------------------------------------------------------------------------------------------------------------
BASELINE (D-SCRIPT)       | 0.5998 | 0.8611 | 0.8106 | 0.5193 | -        | -       
config_1 (Auto)           | 0.8736 | 0.9549 | 0.8742 | 0.7959 | +0.2737 | +0.0937
config_2 (Auto)           | 0.8944 | 0.9608 | 0.9637 | 0.7404 | +0.2946 | +0.0996
config_3 (Auto)           | 0.8783 | 0.9666 | 0.9094 | 0.7563 | +0.2785 | +0.1054
config_1 (Spatial)        | 0.8467 | 0.9513 | 0.8618 | 0.7554 | +0.2469 | +0.0902
config_2 (Spatial)        | 0.8693 | 0.9617 | 0.9074 | 0.7595 | +0.2694 | +0.1006
config_3 (Spatial)        | 0.8357 | 0.9480 | 0.8976 | 0.6739 | +0.2359 | +0.0868
--------------------------------------------------------------------------------------------------------------
🏆 MIGLIORE CONFIGURAZIONE TOTALE SU CANDIDA ALBICANS: config_2 (Auto)
📈 VALORE AUPR: 0.8944
